In [12]:
import papermill as pm
import nbformat
import pandas as pd
import os

SEEDS = [0, 1, 2, 3, 4]

LOSS_NOTEBOOKS = {
    "ArcFace": "test_arcface.ipynb",
    "Triplet": "test_triplet.ipynb",
    "CrossEntropy": "test_crossentropy.ipynb"
}

OUTPUT_DIR = "outputs_loss_functions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []

for loss_name, notebook_path in LOSS_NOTEBOOKS.items():
    for seed in SEEDS:
        output_path = f"{OUTPUT_DIR}/test_run_{loss_name}_{seed}.ipynb"

        pm.execute_notebook(
            f"{OUTPUT_DIR}/{notebook_path}",
            output_path,
            parameters=dict(
                query_ratio=0.2,
                seed=seed
            )
        )

        # read final output cell
        nb = nbformat.read(output_path, as_version=4)
        output_cell = nb.cells[-1]

        output_text = output_cell["outputs"][0]["data"]["text/plain"]
        d = eval(output_text)

        results.append({
            "loss_function": d["loss_function"],
            "seed": d["seed"],
            "accuracy": float(d["accuracy"])
        })

df = pd.DataFrame(results)

summary = (
    df.groupby("loss_function")
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std")
    )
    .reset_index()
)

display(summary.sort_values("mean_accuracy", ascending=False))

Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

Executing:   0%|          | 0/9 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

Executing:   0%|          | 0/16 [00:00<?, ?cell/s]

,loss_function,mean_accuracy,std_accuracy
2,TripletLoss,0.625000,0.036644
0,ArcFace,0.617391,0.019444
1,CrossEntropyLoss,0.587302,0.051434


In [13]:
df.to_csv("loss_functions_results.csv", index=False)